In [1]:
# pip install git+https://github.com/IBM/terratorch.git ##used this code to get latest terratorch download

from Decoder_UNet2D import UNet2D
from Encoder_TerraMind import TerraMindEncoder
from DataLoader import BeforeData
# from Losses import DiceLoss
from monai.losses.dice import DiceLoss # replaced the above loss
from utils import RandomFlipPair, RandomRotationPair, weights, calc_batch_metrics, calc_epoch_metrics, move_to_device, set_seeds, create_writer, save_checkpoint
from DataLoader import MultiModalBeforeAfterDataset



import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# following article: https://www.learnpytorch.io/07_pytorch_experiment_tracking/
from torchinfo import summary
import torch.utils.tensorboard
from torch.utils.tensorboard import SummaryWriter



INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations


# New DataLoader Multimodal 

In [7]:
# -----params----- #
learning_rate = 0.00005
num_classes = 3
num_augmentations = 0
batch_size = 1
num_epochs = 2
ignore_index = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
apply_weight = False
version = "terramind_v1_base"
pretrained = True
modalities = ["S2L2A", "S1GRD"]
set_seeds(22)


# ------Loading in models + data + helper functions --------- #
train_data = MultiModalBeforeAfterDataset(
    modalities = {'S2L2A': (os.getcwd()+ "/Images/Before/S2L2A", os.getcwd()+ "/Images/After/S2L2A"),
                   'S1GRD': (os.getcwd()+ "/Images/Before/S1GRD", os.getcwd()+ "/Images/After/S1GRD")},
    label_dir = os.getcwd() + "/Images/Relabeled",
    split = 'train', 
    num_augmentations= num_augmentations,
    patch_size=224,
    stride = 224,
    add_random_offset=False,
    preload=False)
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

test_data = MultiModalBeforeAfterDataset(
    modalities = {'S2L2A': (os.getcwd()+ "/Images/Before/S2L2A", os.getcwd()+ "/Images/After/S2L2A"),
                   'S1GRD': (os.getcwd()+ "/Images/Before/S1GRD", os.getcwd()+ "/Images/After/S1GRD")},
    label_dir = os.getcwd() + "/Images/Relabeled",
    split = 'test', 
    patch_size=224,
    stride = 224,
    add_random_offset=False,
    preload=False)
test_dataloader = DataLoader(test_data, batch_size=1, shuffle=True, num_workers=2)


if apply_weight == True:
    weights_ = weights(train_dataloader, num_classes=num_classes, ignore_index=ignore_index, device = device)
    criterion = nn.CrossEntropyLoss(weight = weights_, ignore_index=ignore_index)
else:
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)


encoder = TerraMindEncoder(version = version, pretrained = pretrained, modalities = modalities )
decoder = UNet2D(num_classes=num_classes)
for param in encoder.parameters():
    param.requires_grad = False
encoder.to(device)
decoder.to(device)

optimizer = optim.Adam(#list(encoder.parameters()) + 
    list(decoder.parameters()),
     lr=learning_rate)

In [20]:
writer = create_writer("WriterTest", 2)

# ---- Training Loop ---- #
encoder.eval()

best_test_loss = float("inf")
for epoch in range(num_epochs):
    decoder.train() 
    running_train_loss = 0.0
    running_test_loss = 0.0
    TP = FP = FN = TN = 0.0
    
    for x, y in train_dataloader:
        x = move_to_device(x, device)
        y = y.to(device)

        z_before, z_after = encoder(x["before"]), encoder(x["after"]) #
        z_differenced = [after - before for before, after in zip(z_before, z_after)]
        logits = decoder(z_differenced)

        train_loss = criterion(logits, y)
        sz_batch = next(iter(x["before"].values())).size(0)
        running_train_loss += train_loss.item() * sz_batch


        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
              

    epoch_loss = running_train_loss / len(train_data)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f}")

    decoder.eval()
    with torch.no_grad():
        iterations = 0
        val_loss = 0
        for x, y in test_dataloader:
            x = move_to_device(x, device)
            y = y.to(device)

            z_before, z_after = encoder(x["before"]), encoder(x["after"])
            z_differenced = [after - before for before, after in zip(z_before, z_after)]
            logits = decoder(z_differenced)
            
            batch_test_loss = criterion(logits, y)
            sz_batch = next(iter(x["before"].values())).size(0)
            running_test_loss += batch_test_loss.item() * sz_batch

            batch_metrics = calc_batch_metrics(logits, y, ignore_index=0, positive_class = 2, negative_class=1)
            TP, FP, FN, TN = [x + y for x, y in zip((TP, FP, FN, TN), batch_metrics)]

        
        test_loss = running_test_loss/len(test_data)
        print(f"--Test Loss: {test_loss}")


        if test_loss < best_test_loss:
            best_test_loss = test_loss
            save_checkpoint(encoder, decoder, optimizer, epoch, test_loss, {"learning_rate": learning_rate, 
                                                                   "Epochs": num_epochs})



        epoch_metrics = calc_epoch_metrics(TP, FP, FN, TN)
        print(f"--IoU: {epoch_metrics["IoU"]:.4f}\
        - Accuracy: {epoch_metrics["Accuracy"]:.4f}\
        - Precision: {epoch_metrics["Precision"]:.4f}\
        - Recall: {epoch_metrics["Recall"]:.4f}\
        - F1: {epoch_metrics["F1"]:.4f}\n")


    writer.add_scalars("Loss",
                       tag_scalar_dict={"train_loss": epoch_loss,
                                        "test_loss": test_loss},
                        global_step= epoch)
    
    writer.add_scalars("IoU",
                    tag_scalar_dict={"IoU": epoch_metrics['IoU']},
                    global_step= epoch)
    
    writer.add_scalars("Accuracy",
                    tag_scalar_dict={"Accuracy": epoch_metrics['Accuracy']},
                    global_step= epoch)
    writer.add_scalars("Precision",
                    tag_scalar_dict={"Precision": epoch_metrics['Precision']},
                    global_step= epoch)
    writer.add_scalars("Recall",
                    tag_scalar_dict={"Recall": epoch_metrics['Recall']},
                    global_step= epoch)
    writer.add_scalars("F1",
                    tag_scalar_dict={"F1": epoch_metrics['F1']},
                    global_step= epoch)

    # writer.add_scalars("Test_Metrics",
    #                 tag_scalar_dict={"IoU": epoch_metrics["IoU"],
    #                                 "Accuracy": epoch_metrics["Accuracy"],
    #                                 "Precision": epoch_metrics["Precision"],
    #                                 "Recall": epoch_metrics["Recall"],
    #                                 "F1": epoch_metrics["F1"]},
    #                 global_step= epoch)
    writer.close()



[INFO] Created SummaryWriter, saving to: runs/2025-10-192255/WriterTest/2...
Epoch 1/2 - Train Loss: 1.0848
--Test Loss: 1.0694050788879395
✅ Saved checkpoint: checkpoints/model_epoch0_valloss1.0694.pt
--IoU: 0.4669        - Accuracy: 0.8472        - Precision: 0.9327        - Recall: 0.4832        - F1: 0.6366

Epoch 2/2 - Train Loss: 1.0595
--Test Loss: 1.0406127572059631
✅ Saved checkpoint: checkpoints/model_epoch1_valloss1.0406.pt
--IoU: 0.4090        - Accuracy: 0.8848        - Precision: 0.9250        - Recall: 0.4231        - F1: 0.5806



In [33]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 34630), started 3 days, 22:32:45 ago. (Use '!kill 34630' to kill it.)

# Old and Trusty Data Loader

In [ ]:
# -----params----- #
learning_rate = 0.0005
num_classes = 3
num_augmentations = 6
batch_size = 3
num_epochs = 5
ignore_index = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
apply_weight = False
version = "terramind_v1_base"
pretrained = True
modalities = ["S2L2A"]

# ------Loading in models + data + helper functions --------- #
train_data = BeforeData(before_dir = os.getcwd()+ "/Images/Before",
                     after_dir = os.getcwd()+ "/Images/After",
                     label_dir = os.getcwd() + "/Images/Relabeled",
                     split = 'train', 
                     num_augmentations= num_augmentations)
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

test_data = BeforeData(before_dir = os.getcwd()+ "/Images/Before",
                     after_dir = os.getcwd()+ "/Images/After",
                     label_dir = os.getcwd() + "/Images/Relabeled",
                     split = 'test')
test_dataloader = DataLoader(test_data, batch_size=1, shuffle=True, num_workers=2)


if apply_weight == True:
    weights_ = weights(train_dataloader, num_classes=num_classes, ignore_index=ignore_index, device = device)
    criterion = nn.CrossEntropyLoss(weight = weights_, ignore_index=ignore_index)
else:
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)


encoder = TerraMindEncoder(version = version, pretrained = pretrained, modalities = modalities )
decoder = UNet2D(num_classes=num_classes)
for param in encoder.parameters():
    param.requires_grad = False
encoder.to(device)
decoder.to(device)

optimizer = optim.Adam(#list(encoder.parameters()) + 
    list(decoder.parameters()),
     lr=learning_rate)

# ---- Training Loop ---- #
encoder.eval()
decoder.train()

train_losses = []
test_losses = []
metrics = []
for epoch in range(num_epochs):
    decoder.train() 
    running_train_loss = 0.0
    running_test_loss = 0.0
    TP = FP = FN = TN = 0.0
    
    for (x_before, x_after), y in train_dataloader:
        x_before, x_after, y = x_before.to(device), x_after.to(device), y.to(device)
        z_before = encoder(x_before)
        z_after = encoder(x_after)
        z_differenced = [after - before for before, after in zip(z_before, z_after)]
        logits = decoder(z_differenced)

        loss = criterion(logits, y)
        running_train_loss += loss.item() * x_before.size(0)        

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
              

    epoch_loss = running_train_loss / len(train_data)
    train_losses.append(epoch_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f}")

    decoder.eval()
    with torch.no_grad():
        iterations = 0
        val_loss = 0
        for (x_before, x_after), y in test_dataloader:
            x_before, x_after, y = x_before.to(device), x_after.to(device), y.to(device)
            z_before, z_after = encoder(x_before), encoder(x_after)
            z_differenced = [after - before for before, after in zip(z_before, z_after)]
            logits = decoder(z_differenced)
            
            test_loss = criterion(logits, y)
            running_test_loss += test_loss.item() * x_before.size(0)

            batch_metrics = calc_batch_metrics(logits, y, ignore_index=0, positive_class = 2, negative_class=1)
            TP, FP, FN, TN = [x + y for x, y in zip((TP, FP, FN, TN), batch_metrics)]

        
        final_test_loss = running_test_loss/len(test_data)
        test_losses.append(final_test_loss)
        print(f"--Test Loss: {test_loss.item()}")

        epoch_metrics = calc_epoch_metrics(TP, FP, FN, TN)
        metrics.append(epoch_metrics)
        print(f"-- Accuracy: {metrics[-1]["accuracy"]:.4f}\
        - Precision: {metrics[-1]["precision"]:.4f}\
        - Recall: {metrics[-1]["recall"]:.4f}\
        - F1: {metrics[-1]["F1"]:.4f}\
        - IoU: {metrics[-1]["IoU"]:.4f}\n")

In [ ]:
plt.plot(train_losses, label = "train")
plt.plot(test_losses, label = "test")
plt.legend()
plt.show()

print(len(train_data))

In [ ]:

#plot Loss and Accuracy
# fig, ax1 = plt.subplots()

# # Plot losses on the left y-axis
# ax1.plot(losses, color="tab:red", label="Loss")
# ax1.set_xlabel("Epoch")
# ax1.set_ylabel("Loss", color="tab:red")
# ax1.tick_params(axis="y", labelcolor="tab:red")

# # Create a second y-axis that shares the same x-axis
# ax2 = ax1.twinx()
# ax2.plot(accuracies, color="tab:blue", label="Accuracy")
# ax2.set_ylabel("Accuracy", color="tab:blue")
# ax2.tick_params(axis="y", labelcolor="tab:blue")

# # Add legends
# fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)

# plt.title("Loss and Accuracy over Epochs")
# plt.show()

In [ ]:
for i in test_dataloader:
    b, a, l = i[0][0], i [0][1], i[1]

l = l.flatten()
plt.figure(figsize=(8,5))
plt.hist(l, bins=range(l.min(),l.max()+2), align='left', rwidth=0.8, color="skyblue", edgecolor="black")
plt.xlabel("Integer Value")
plt.ylabel("Frequency")
plt.title("Distribution of Tensor Values")
plt.show()


predictions = torch.argmax(logits, dim=1)
p = predictions.flatten()
plt.figure(figsize=(8,5))
plt.hist(p, bins=range(p.min(),p.max()+2), align='left', rwidth=0.8, color="skyblue", edgecolor="black")
plt.xlabel("Integer Value")
plt.ylabel("Frequency")
plt.title("Distribution of Tensor Values")
plt.show()

cm = confusion_matrix(y.cpu().numpy().ravel(),predictions.cpu().numpy().ravel())
disp = ConfusionMatrixDisplay(confusion_matrix=cm)#, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap=plt.cm.Blues) # You can choose different colormaps
plt.title('Confusion Matrix')
plt.show()



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = 3
ignore_index = 0
apply_weight = True

# encoder = TerraMindEncoder()
# decoder = UNet2D(num_classes = num_classes)
# encoder.to(device)
# decoder.to(device)

dataset = BeforeData(before_dir = os.getcwd()+ "/Images/Before",
                     after_dir = os.getcwd()+ "/Images/After",
                     label_dir = os.getcwd() + "/Images/Relabeled",
                     split = 'test')


dataloader = DataLoader(dataset)

if apply_weight == True:
    weights_ = weights(dataloader, num_classes=num_classes, ignore_index=ignore_index, device = device)
    criterion = nn.CrossEntropyLoss(weight = weights_, ignore_index=ignore_index)
else:
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

# ---- Test Loop ----
loss = []
accuracies = []
running_loss = 0.0

encoder.eval()
decoder.eval()
with torch.no_grad():  # no gradients for inference
    test_loss = 0
    for (x_before, x_after), y in dataloader:
        x_before, x_after, y = x_before.to(device), x_after.to(device), y.to(device)

        z_before = encoder(x_before)
        z_after = encoder(x_after)
        z_differenced = [after - before for before, after in zip(z_before, z_after)]

        logits = decoder(z_differenced)

        metrics = calculate_metrics(logits, y, mask)

        
        loss = criterion(logits, y)
        test_loss += loss


test_accuracy = total_correct/total_pixels
print("Loss:", test_loss)
print("Accuracy:", test_accuracy)    


# plt.plot(losses, label = "Loss")
# plt.plot(accuracies, label = "Accuracy")
# plt.legend()
# plt.show()

cm = confusion_matrix(y.cpu().numpy().ravel(), predictions.cpu().numpy().ravel())
disp = ConfusionMatrixDisplay(confusion_matrix=cm)#, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap=plt.cm.Blues) # You can choose different colormaps
plt.title('Confusion Matrix')
plt.show()

print("Accuracy:", accuracies)

In [ ]:
# predictions[y==0] = 0 


# x_before_img = x_before[0]
# x_after_img  = x_after[0] 
# y_img        = y[0]       
# pred_img     = predictions[0]

# # Setup 2x2 grid with constrained layout
# fig, axes = plt.subplots(2, 2, figsize=(18, 14), constrained_layout=True)

# # --- Before image (RGB: bands 4,3,2) ---
# rgb_before = x_before_img[[3, 2, 1], :, :].permute(1, 2, 0)
# rgb_before = (rgb_before - rgb_before.min()) / (rgb_before.max() - rgb_before.min())
# axes[0, 0].imshow(rgb_before.cpu())
# axes[0, 0].set_title("Before (RGB)", fontsize=18, pad=12)
# axes[0, 0].axis("off")

# # --- After image (RGB: bands 4,3,2) ---
# rgb_after = x_after_img[[3, 2, 1], :, :].permute(1, 2, 0)
# rgb_after = (rgb_after - rgb_after.min()) / (rgb_after.max() - rgb_after.min())
# axes[0, 1].imshow(rgb_after.cpu())
# axes[0, 1].set_title("After (RGB)", fontsize=18, pad=12)
# axes[0, 1].axis("off")

# # --- Ground truth labels ---
# im1 = axes[1, 0].imshow(y_img.cpu(), cmap="tab10", interpolation="nearest")
# axes[1, 0].set_title("Labels (Ground Truth)", fontsize=18, pad=12)
# axes[1, 0].axis("off")

# # --- Predictions ---
# im2 = axes[1, 1].imshow(pred_img.cpu(), cmap="tab10", interpolation="nearest")
# axes[1, 1].set_title("Predictions", fontsize=18, pad=12)
# axes[1, 1].axis("off")

# # Single shared colorbar (placed to the right of all plots)
# cbar = fig.colorbar(im2, ax=axes, shrink=0.8, location="right")
# cbar.set_label("Class IDs", fontsize=16)

# plt.show()